# 🏆 Churn Prediction: Master Production Pipeline (V5.2 - FINAL PRO)
**Core Priority**: EDA + Data Profiling + RECALL + SHAP + Gender Equity Audit.

In [ ]:
# 1. Setup
!nvidia-smi
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub -q

import torch
HAS_GPU = torch.cuda.is_available()

import opendatasets as od
import os, json, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, f1_score, confusion_matrix, roc_curve, auc, classification_report
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from mlflow.models import infer_signature

mlflow.sklearn.autolog(log_models=False)

In [ ]:
# 2. Auth & Data Loading with Profiling
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

def clean_dataset(df):
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid', 'last_interaction'] if c in df.columns])
    if 'age' in df.columns: df = df[(df['age'] >= 18) & (df['age'] <= 120)]
    for col in [c for c in ['total_spend', 'totalcharges'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['churn'])

if not os.path.exists("customer-churn-dataset"):
    od.download("https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset")

df_raw = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-training-master.csv"))
df_test_final = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-testing-master.csv"))
df_train, df_val = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=df_raw['churn'])

print("\n--- 📊 DATA PROFILE ---")
print(f"Training Set: {df_train.shape[0]} rows, {df_train.shape[1]} features")
print(f"Validation Set: {df_val.shape[0]} rows")
print(f"Test Set: {df_test_final.shape[0]} rows")
print(f"Training Churn Rate: {df_train['churn'].mean():.2%}")
print(f"Testing Churn Rate: {df_test_final['churn'].mean():.2%}")
print("\n--- Feature Types ---")
print(df_train.dtypes)
print("\n--- Missing Values ---")
print(df_train.isnull().sum()[df_train.isnull().sum() > 0] if df_train.isnull().sum().any() else "No missing values.")
print("✅ Data Loading & Profiling Done.")

## 3. Exploratory Data Analysis (EDA)
Visualizing distribution and key correlations.

In [ ]:
plt.figure(figsize=(18, 5))

# 3.1 Imbalance Check
plt.subplot(1, 3, 1)
sns.countplot(x='churn', data=df_train, palette='magma')
plt.title("Churn Distribution (Imbalance)")

# 3.2 Correlation Analysis
plt.subplot(1, 3, 2)
corr = df_train.select_dtypes(include=[np.number]).corr()['churn'].sort_values().drop('churn')
corr.plot(kind='barh', color='darkcyan')
plt.title("Feature Correlation with Churn")

# 3.3 Spend vs Churn
plt.subplot(1, 3, 3)
sns.boxplot(x='churn', y='total_spend', data=df_train, palette='Set2')
plt.title("Total Spend vs Churn")

plt.tight_layout(); plt.show()

## 4. Enhanced Training & Responsible AI Audit

In [ ]:
def run_master_experiment(df_tr, df_va, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), X_tr.select_dtypes(include=[np.number]).columns.tolist()),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), X_tr.select_dtypes(include=['object']).columns.tolist())
    ])
    
    if model_type == "xgboost":
        pw = (y_tr == 0).sum() / (y_tr == 1).sum()
        clf = XGBClassifier(scale_pos_weight=pw, device='cuda' if HAS_GPU else 'cpu', tree_method='hist')
        params = {'clf__max_depth': [3, 5]}
    elif model_type == "random_forest":
        clf = RandomForestClassifier(class_weight='balanced', random_state=42)
        params = {'clf__max_depth': [10, 20]}
    else:
        clf = LogisticRegression(class_weight='balanced', max_iter=2000)
        params = {'clf__C': [0.1, 1.0]}
        
    mlflow.set_experiment("Churn_Final_Professional_Audit")
    with mlflow.start_run(run_name=f"NB_{model_type.upper()}"):
        print(f"\n🔥 STARTING: {model_type.upper()}")
        pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
        search = RandomizedSearchCV(pipe, params, n_iter=2, cv=2, scoring='recall', verbose=2)
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        
        # 1. Gender Equity Drilldown
        y_pred = model.predict(X_te)
        results_df = df_te.copy(); results_df['pred'] = y_pred
        gender_stats = results_df.groupby('gender').agg(Actual=('churn', 'mean'), Predicted=('pred', 'mean'))
        print(f"\n📊 Gender Churn Comparison ({model_type.upper()}):")
        print(gender_stats)
        
        # 2. Visuals
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        gender_stats.plot(kind='bar', ax=ax1, title=f"Gender Equity Tracking: {model_type}")
        y_probs = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else y_pred
        fpr, tpr, _ = roc_curve(y_te, y_probs)
        ax2.plot(fpr, tpr, label=f'AUC={auc(fpr,tpr):.2f}'); ax2.set_title(f"ROC Stability: {model_type}"); ax2.legend()
        plt.show()
        
        # 3. SHAP (Professional Setting)
        try:
            sample = X_te.sample(min(100, len(X_te)))
            X_transformed = pd.DataFrame(model.named_steps['pre'].transform(sample), columns=model.named_steps['pre'].get_feature_names_out())
            explainer = shap.LinearExplainer(model.named_steps['clf'], X_transformed) if model_type == "logistic_regression" else shap.Explainer(model.named_steps['clf'], X_transformed)
            shap_values = explainer(X_transformed)
            plt.figure(figsize=(10, 8)); shap.summary_plot(shap_values, X_transformed, show=False, max_display=20)
            plt.savefig(f"shap_{model_type}.png"); mlflow.log_artifact(f"shap_{model_type}.png"); plt.show()
        except Exception as e: print(f"⚠️ SHAP Skip: {e}")

        mlflow.sklearn.log_model(model, "model", registered_model_name=f"CustomerChurnModel_{model_type}")
        print(f"🏆 {model_type.upper()} REGISTERED SUCCESS.")

for m in ["xgboost", "random_forest", "logistic_regression"]:
    run_master_experiment(df_train, df_val, df_test_final, m)